In [10]:
"""
Step 1 of the glass pipeline: read a range of frames from the raw video
on the external drive, crop to a pixel region, and write out a lossless
working copy -- without loading all frames into memory at once, since
the source files are huge (~200GB).

This does NOT denoise, compress, or otherwise alter pixel values --
it only crops (spatial) and trims (temporal). Time-average background
subtraction happens in a later step, on this cropped output.
"""

import cv2
import os

# ----------------------------------------------------------------------
# 1. CONFIG -- edit these values
# ----------------------------------------------------------------------

INPUT_PATH = "/Volumes/Expansion/recordings/theo_seth_1 26-08-10 17-56-30.avi"

# Frame range (inclusive start, exclusive end -- Python-style slicing)
FRAME_START = 0
FRAME_END = 4000

# Crop bounds, in pixels. (0,0) is the top-left corner of the raw frame.
X_MIN = 128  # <-- fill in
X_MAX = 1700  # <-- fill in
Y_MIN = 1458  # <-- fill in
Y_MAX = 2890  # <-- fill in

# Output: same folder as the input file, new filename
OUTPUT_DIR = os.path.dirname(INPUT_PATH)
OUTPUT_FILENAME = "theo_seth_1_cropped_0-4000.avi"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)

# ----------------------------------------------------------------------
# 2. OPEN THE SOURCE VIDEO
# ----------------------------------------------------------------------

def main():
    if X_MIN is None or X_MAX is None or Y_MIN is None or Y_MAX is None:
        raise ValueError(
            "Set X_MIN, X_MAX, Y_MIN, Y_MAX before running this script."
        )

    cap = cv2.VideoCapture(INPUT_PATH)
    if not cap.isOpened():
        raise IOError(f"Could not open video: {INPUT_PATH}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    native_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    native_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"Source video: {total_frames} frames, {native_width}x{native_height} px")

    if FRAME_END > total_frames:
        raise ValueError(
            f"FRAME_END={FRAME_END} exceeds total frames in video ({total_frames})"
        )

    crop_width = X_MAX - X_MIN
    crop_height = Y_MAX - Y_MIN
    print(f"Cropping to: x[{X_MIN}:{X_MAX}] y[{Y_MIN}:{Y_MAX}]  "
          f"({crop_width}x{crop_height} px)")
    print(f"Frame range: {FRAME_START} to {FRAME_END} "
          f"({FRAME_END - FRAME_START} frames)")

    # ------------------------------------------------------------------
    # 3. SET UP THE OUTPUT WRITER (lossless FFV1 codec)
    # ------------------------------------------------------------------

    fourcc = cv2.VideoWriter_fourcc(*"FFV1")
    fps = cap.get(cv2.CAP_PROP_FPS) or 1.0  # your video is 1 fps
    writer = cv2.VideoWriter(
        OUTPUT_PATH, fourcc, fps, (crop_width, crop_height), isColor=False
    )
    if not writer.isOpened():
        raise IOError(f"Could not open VideoWriter for: {OUTPUT_PATH}")

    # ------------------------------------------------------------------
    # 4. STREAM THROUGH FRAMES: read -> grayscale -> crop -> write
    #    We never hold more than one frame in memory at a time.
    # ------------------------------------------------------------------

    # Jump directly to FRAME_START instead of reading through frames we skip
    cap.set(cv2.CAP_PROP_POS_FRAMES, FRAME_START)

    first_frame_preview = None
    last_frame_preview = None
    n_written = 0

    for i in range(FRAME_START, FRAME_END):
        ret, frame = cap.read()
        if not ret:
            print(f"WARNING: failed to read frame {i}, stopping early.")
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        cropped = gray[Y_MIN:Y_MAX, X_MIN:X_MAX]

        writer.write(cropped)
        n_written += 1

        if i == FRAME_START:
            first_frame_preview = cropped.copy()
        last_frame_preview = cropped.copy()

        if i % 500 == 0:
            print(f"  processed frame {i}")

    cap.release()
    writer.release()

    print(f"\nDone. Wrote {n_written} frames to:\n  {OUTPUT_PATH}")

    # ------------------------------------------------------------------
    # 5. SANITY CHECK -- save first/last frame previews as .png
    #    so you can visually confirm the crop and check for corruption
    # ------------------------------------------------------------------

    preview_dir = os.path.join(OUTPUT_DIR, "crop_previews")
    os.makedirs(preview_dir, exist_ok=True)
    cv2.imwrite(os.path.join(preview_dir, "first_frame.png"), first_frame_preview)
    cv2.imwrite(os.path.join(preview_dir, "last_frame.png"), last_frame_preview)
    print(f"Saved preview frames to: {preview_dir}")


if __name__ == "__main__":
    main()

Source video: 1728475 frames, 4096x3000 px
Cropping to: x[128:1700] y[1458:2890]  (1572x1432 px)
Frame range: 0 to 4000 (4000 frames)
  processed frame 0
  processed frame 500
  processed frame 1000
  processed frame 1500
  processed frame 2000
  processed frame 2500
  processed frame 3000
  processed frame 3500

Done. Wrote 4000 frames to:
  /Volumes/Expansion/recordings/theo_seth_1_cropped_0-4000.avi
Saved preview frames to: /Volumes/Expansion/recordings/crop_previews
